# Baseline Training

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import torch

In [2]:
import torch.nn as nn
from torch.utils.data import Subset, DataLoader

In [3]:
from helpers.data_funcs import make_index #need to restart kernel to get these updated!!
from helpers.defs import IcepackDataset
from helpers.defs import read_mod_file

ModuleNotFoundError: No module named 'helpers'

## Config

In [ ]:
RUN_NAME = "ksno16"
SPLIT_NAME = "split1"
EXPERIMENT_NAME = "MLP_3layers"
SEED = 42

In [ ]:
IN_DIR = Path(f"/projects/dagr5998/cleaned_data/01icepack_array_output/{RUN_NAME}_daily")
SPLIT_DIR = Path(f"/projects/dagr5998/cleaned_data/02icepack_split_output/{RUN_NAME}_split/{SPLIT_NAME}")
OUT_DIR = Path(f"/projects/dagr5998/cleaned_data/03icepack_experiment_output/{EXPERIMENT_NAME}")
torch.manual_seed(SEED)

#### Loading Data

In [ ]:
X = np.load(IN_DIR / "X.npy") # input array
Y = np.load(IN_DIR / "Y.npy") # output array
sets = np.load(IN_DIR / "sets.npy") # changed parameter set index
parameterizations = np.load(IN_DIR / "parameterizations.npy") # changed parameterizations
locations = np.load(IN_DIR/"locations.npy") 

In [ ]:
mu,sd = np.load(SPLIT_DIR / 'mu.npy'), np.load (SPLIT_DIR/'sd.npy')
mu,sd = torch.from_numpy(mu), torch.from_numpy(sd)
train_index = np.load (SPLIT_DIR/"train_index.npy")
train_ind = np.load (SPLIT_DIR/"train_ind.npy")
val_index = np.load (SPLIT_DIR/"val_index.npy")
val_ind = np.load (SPLIT_DIR/"val_ind.npy")

In [ ]:
with open(IN_DIR / 'config.json') as f:
    config = json.load(f)
with open(SPLIT_DIR / 'split_config.json') as f:
    split_config = json.load(f)
var_ind_list = split_config["var_ind_list"]
Xt = torch.from_numpy(X[:, :, var_ind_list].copy())
Yt = torch.from_numpy(Y)

In [ ]:
training_vars = split_config["training_vars"]

In [ ]:
assert Xt.shape[2] == len(training_vars) #same channels
assert Yt.shape[0] == Xt.shape[0] #same number of runs
assert Path(split_config["in_path"])== (IN_DIR) #same data
assert split_config["n_test"] == len(val_index)
assert split_config["n_train"] <= len(train_index)

In [ ]:
input_window = split_config["input_window_timesteps"]

In [ ]:
train_ds = IcepackDataset(Xt, Yt, train_index, input_window,mu,sd)
#test_ds = IcepackDataset(Xt, Yt, test_index,  input_window,mu,sd)
val_ds = IcepackDataset(Xt, Yt, val_index, input_window, mu,sd)

#### Non-ML Checks

In [ ]:
baseline = Yt[list(train_ind)].mean(dim = 0)
baseline

In [ ]:
plt.plot(np.linspace(0,Yt[test_ind].shape[0],Yt[test_ind].shape[0]), [baseline[0]]*Yt[test_ind].shape[0], color = "#565A5C")
plt.scatter(range((Yt[test_ind].shape[0])),Yt[test_ind][:,0], color = "#CFB87C")
plt.title ("Uniform Guess")
plt.ylabel("Ksno Value")
plt.xlabel("Run")
plt.show()

## Testing Neural Net Training Loop

In [ ]:
overfitting_set = {}
for i, l in enumerate(train_index):
    overfitting_set[sets[l[0]]]= i          # last training sample from each set
picks = list(overfitting_set.values())

In [ ]:
xbatch, ybatch = next(iter(DataLoader(Subset(train_ds, picks), batch_size=5)))
y_mu, y_sd = Yt[train_ind].mean(0), Yt[train_ind].std(0)
ybatch_01 = (ybatch-y_mu)/y_sd
print(xbatch.shape, ybatch.shape)

In [ ]:
loss_fn0 = nn.MSELoss()
model0 = nn.Sequential(nn.Flatten(), nn.Linear(6205, 1))
optimizer0 = torch.optim.Adam(model0.parameters(), lr=1e-3)

In [ ]:
loss_list = []
for n in range(1000):
    y_pred = model0(xbatch)
    loss = loss_fn0(y_pred, ybatch_01)
    optimizer0.zero_grad()
    loss.backward()
    optimizer0.step()
    loss_list.append(loss.item())

In [ ]:
print((ybatch_01*y_sd + y_mu)- (model0(xbatch)*y_sd + y_mu)) # this is just asking whether or not the model can memorize 5 years, it would be weird if it couldnt

In [ ]:
plt.semilogy(loss_list)
plt.xlabel("Steps")
plt.ylabel("Log Error")
plt.title("NN MSE over training")
plt.show()

# First Neural Networks!

In [ ]:
def train(model, dataloader, optimizer):
    # one training pass through the loader
    model.train() #sets to training mode
    loss_tot = 0
    n=0
    for xbatch, ybatch in dataloader:
        ybatch_std = (ybatch-y_mu)/y_sd 
        loss = loss_fn(model(xbatch), ybatch_std)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_tot += loss.item() * len(xbatch)
        n += len(xbatch)
    return loss_tot/n #returns mse
    
@torch.no_grad() # don't train on benchmark data
def validate(model, dataloader):
    model.eval() #sets to evaluate mode
    loss_tot = 0
    n=0
    for xbatch, ybatch in dataloader:
        ybatch_std = (ybatch-y_mu)/y_sd 
        loss = loss_fn(model(xbatch), ybatch_std)
        loss_tot += loss.item() * len(xbatch)
        n += len(xbatch)
    return loss_tot/n

@torch.no_grad()
def evaluate(model, dataset):
    model.eval()
    x_in = torch.stack([dataset[i][0] for i in range(len(dataset))])
    return (model(x_in)*y_sd + y_mu)

### Linear Neural Net (Test)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader   = DataLoader(test_ds,   batch_size=64, shuffle=False)
y_mu, y_sd = Yt[train_ind].mean(0), Yt[train_ind].std(0)

In [ ]:
loss_fn = nn.MSELoss()
model = nn.Sequential(nn.Flatten(), nn.Linear(6205, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_history = []
test_history = []

for epoch in range(500):
    train_history.append(train(model, train_loader, optimizer))
    test_history.append(validate(model, test_loader))

In [ ]:
plt.semilogy(train_history, label = "training dataset error")
plt.semilogy(test_history, label = "unseen dataset error")
plt.xlabel("Epochs")
plt.ylabel("Log Error")
plt.legend()
plt.title("Single Linear Neural Network Layer")
plt.show()

## Basic Neural Net 

In [ ]:
loss_fn = nn.MSELoss()
model = nn.Sequential(nn.Flatten(), nn.Linear(6205, 64), nn.ReLU(), nn.Linear(64,1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_history_3 = []
test_history_3 = []

for epoch in range(300):
    train_history_3.append(train(model, train_loader, optimizer))
    test_history_3.append(validate(model, test_loader))

In [ ]:
plt.semilogy(train_history_3, label = "training dataset error")
plt.semilogy(test_history_3, label = "unseen dataset error")
plt.xlabel("Epochs")
plt.ylabel("Log Error")
plt.legend()
plt.title("Three Layer Neural Network Error")
plt.show()

In [ ]:
guesses_train =  evaluate(model,train_ds)
truth_train =   Yt[train_ds.index[:,0]]

guesses_test =  evaluate(model,test_ds)
truth_test =   Yt[test_ds.index[:,0]]
loc_test = locations[test_index[:, 0]]
loc_train = locations[train_index[:, 0]]
loc_list = ["centralarctic", "coastalcanada", "sibchuk", "barents"]
loc_index = [0,1,2,3]
loc_list = [ loc_list[i] for i in loc_index ]

for loc in loc_list:
    mask = (loc_train ==loc)
    plt.scatter(truth_train[mask], guesses_train[mask], alpha=0.5, label=loc)


plt.plot(np.linspace(0.14, 0.43,100), np.linspace(0.14, 0.43,100), ls = "--", color = "black")
plt.legend()
plt.xlabel("True ksno")
plt.ylabel("Estimated ksno")
plt.title("Training Data true vs estimated ksno")
plt.show()

for loc in loc_list:
    mask = (loc_test ==loc)
    plt.scatter(truth_test[mask], guesses_test[mask], alpha=0.5, label=loc)
plt.plot(np.linspace(0.14, 0.43,100), np.linspace(0.14, 0.43,100), ls = "--", color = "black")
plt.legend()
plt.title("Testing Data true vs estimated ksno")
plt.xlabel("True ksno")
plt.ylabel("Estimated ksno")
plt.show()

## Testing Adding Neural Net Layers

In [ ]:
loss_fn = nn.MSELoss()
model = nn.Sequential(nn.Flatten(), nn.Linear(6205, 256), nn.ReLU(), nn.Linear(256,32), nn.ReLU(), nn.Linear(32,1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_history_5 = []
test_history_5 = []

for epoch in range(300):
    train_history_5.append(train(model, train_loader, optimizer))
    test_history_5.append(validate(model, test_loader))

In [ ]:
loss_fn = nn.MSELoss()
model = nn.Sequential(nn.Flatten(), nn.Linear(6205, 1024), nn.ReLU(), nn.Linear(1024, 256), nn.ReLU(), nn.Linear(256,32), nn.ReLU(), nn.Linear(32,1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_history_7 = []
test_history_7 = []

for epoch in range(300):
    train_history_7.append(train(model, train_loader, optimizer))
    test_history_7.append(validate(model, test_loader))

In [ ]:
plt.semilogy(train_history_3, label = "testing dataset error, 3 layers")
plt.semilogy(train_history_5, label = "testing dataset error, 5 layers")
plt.semilogy(train_history_7, label = "testing dataset error, 7 layers")
plt.xlabel("Epochs")
plt.ylabel("Log Error")
plt.legend()
plt.title("Three Layer Neural Network Error")
plt.show()

In [ ]:
print(f"Final testing rmse:")
print(f"3 layers: {np.sqrt(test_history_3[-1])}")
print(f"5 layers: {np.sqrt(test_history_5[-1])}")
print(f"7 layers: {np.sqrt(test_history_7[-1])}")

 ## Model Issues

In [ ]:
guesses_train =  evaluate(model,train_ds)
truth_train =   Yt[train_ds.index[:,0]]

guesses_test =  evaluate(model,test_ds)
truth_test =   Yt[test_ds.index[:,0]]

#guesses_val =  evaluate(model,_ds)
#truth_val =   Yt[test_ds.index[:,0]]

In [ ]:
split_config

In [ ]:
type(guesses_test)

In [ ]:
crit = nn.MSELoss()
rmse = torch.sqrt(crit(truth_test,guesses_test))

In [ ]:
def rmse(tensor1, tensor2):
    crit = nn.MSELoss()
    return torch.sqrt(crit(tensor1,tensor2))

In [ ]:
rmse(truth_test, guesses_test)

In [ ]:
guesses_train =  evaluate(model,train_ds)
truth_train =   Yt[train_ds.index[:,0]]
plt.rcParams['figure.figsize'] = [14, 6]
guesses_test =  evaluate(model,test_ds)
truth_test =   Yt[test_ds.index[:,0]]
loc_test = locations[test_index[:, 0]]
loc_train = locations[train_index[:, 0]]
loc_list = ["centralarctic", "coastalcanada", "sibchuk", "barents"]
loc_index = [0,1,2,3]
loc_list = [ loc_list[i] for i in loc_index ]
plt.subplot(1,2, 1)
for loc in loc_list:
    mask = (loc_train ==loc)
    plt.scatter(truth_train[mask], guesses_train[mask], alpha=0.5, label=loc)


plt.plot(np.linspace(0.14, 0.43,100), np.linspace(0.14, 0.43,100), ls = "--", color = "black")
plt.axhline(baseline, alpha =.25, color = "grey")
plt.legend()
plt.xlabel("True ksno")
plt.ylabel("Estimated ksno")
plt.title(f"Training Data true vs estimated ksno, RMSE = {rmse(guesses_train, truth_train):.4f}")

plt.subplot(1, 2, 2)
for loc in loc_list:
    mask = (loc_test ==loc)
    plt.scatter(truth_test[mask], guesses_test[mask], alpha=0.5, label=loc)
plt.plot(np.linspace(0.14, 0.43,100), np.linspace(0.14, 0.43,100), ls = "--", color = "black")
plt.axhline(baseline, alpha =.25, color = "grey")
plt.legend()
plt.title(f"Testing Data true vs pred ksno, RMSE = {rmse(guesses_test, truth_test):.4f}")
plt.xlabel("True ksno")
plt.ylabel("Estimated ksno")
plt.show()

In [ ]:
vicen_idx = [split_config["training_vars"].index(f"vicen_ncat{i}") for i in range(1,6)]
start = (2000 - config["first_year"])*365

In [ ]:
Xt.shape[1]/365

In [ ]:
def ice_days(index, window =365, cutoff = 0.15):
    # 0.15 from DAlooza talk
    aicen_idx = [split_config["training_vars"].index(f"aicen_ncat{i}") for i in range(1,6)]
    days_list = []
    for run,start in index:
        days_list.append(int(sum(Xt[run,start:start+window,aicen_idx].sum(1)>0.15)))
    return np.array(days_list)

In [ ]:
ice_test = ice_days(test_index)

In [ ]:
for loc in loc_list:
    mask = (loc_test ==loc)
    plt.scatter(ice_test[mask], torch.abs(guesses_test[mask] - truth_test[mask]), alpha =.3, label = loc)
plt.legend()
plt.title("Error in Ksno based on presence of ice")
plt.xlabel("Days of ice in input window")
plt.ylabel("Error in ksno")
plt.show()

In [ ]:
ca14 = Xt[8,:,vicen_idx]
ca43 = Xt[15,:,vicen_idx]
plt.plot(np.linspace(0,16060,16060), ca14.sum(1), label = "ksno =.14")
plt.plot(np.linspace(0,16060,16060), ca43.sum(1), label = "ksno = .43")
plt.legend()
plt.ylabel("ice volume")
plt.xlabel("days since jan 1 1980")
plt.title("2000 with 2 different ksno values in the central arctic")
plt.show()

In [ ]:
b14 = Xt[0,:,vicen_idx]
b43 = Xt[7,:,vicen_idx]
plt.plot(np.linspace(0,16060,16060), b14.sum(1), label = "ksno =.14")
plt.plot(np.linspace(0,16060,16060), b43.sum(1), label = "ksno = .43")
plt.legend()
plt.ylabel("ice volume")
plt.xlabel("days since jan 1 1980")
plt.title("2 different ksno values in the barents")
plt.show()

In [ ]:
cc14 = Xt[16,:,vicen_idx]
cc43 = Xt[23,:,vicen_idx]
plt.plot(np.linspace(0,16060,16060), cc14.sum(1), label = "ksno =.14")
plt.plot(np.linspace(0,16060,16060), cc43.sum(1), label = "ksno = .43")
plt.legend()
plt.ylabel("ice volume")
plt.xlabel("days since jan 1 1980")
plt.title("2 different ksno values in coastal canada")
plt.show()

In [ ]:
sc14 = Xt[24,:,vicen_idx]
sc43 = Xt[31,:,vicen_idx]
plt.plot(np.linspace(0,16060,16060), sc14.sum(1), label = "ksno =.14")
plt.plot(np.linspace(0,16060,16060), sc43.sum(1), label = "ksno = .43")
plt.legend()
plt.ylabel("ice volume")
plt.xlabel("days since jan 1 1980")
plt.title("2 different ksno values in the sibchuk")
plt.show()